In [1]:
# !pip install openai==0.28

# # for testing the model
# set HTTPS_PROXY=czgdcsdwankerb1.czds.bz:8080
# pip install --upgrade openai
# set HTTPS_PROXY=

# # for data upload
# set HTTPS_PROXY=czgdcsdwankerb1.czds.bz:8080
# pip install openai==0.28.1
# set HTTPS_PROXY=

In [2]:
# import ast
# import pandas as pd
# import json
import tiktoken
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
import openai
# import time
# import os
from dotenv import load_dotenv, find_dotenv

import ast
import pandas as pd
import json
import time
import os

In [3]:
_ = load_dotenv(find_dotenv())  # Read local .env file

open_api_engine= "oai-gst-d-usnc-01"
openai.api_type = "azure"
openai.api_base = "https://oai-gst-d-usnc-01.openai.azure.com/"
openai.api_version = os.getenv('open_api_version')
openai.api_key = "4b216a50a8704d0d9a234c4e4d25b545"

In [4]:
# version = "27_01_25" # replace in all cells

In [5]:
td_27_01_25 = pd.read_excel("data/Training_Data_27_01_25/Training_Data_27_01_25.xlsx")
td_27_01_25.head()

,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5


In [6]:
DEFAULT_SYSTEM_PROMPT = "Act as an NER model trained on the data corpus of 'Celanese' which is a global chemical leader in the production of differentiated chemistry solutions and specialty materials used in most major industries and consumer applications. Ensure the output is a structured dictionary format."
# DEFAULT_SYSTEM_PROMPT = """Act as an NER model trained on the data corpus of 'Celanese' which is a global chemical leader in the production of differentiated chemistry solutions and specialty materials used in most major industries and consumer applications. Ensure the output is structured like the below given sample dictionary format, values added here are just for reference:
# '''
# {"GRADE": ["celcon m90"], "APPLICATION": ["speaker grill"], "BRAND": ["hostaform"], "POLYMER": ["pps", "pom"], "PROPERTY": [{"property_name": "density", "modifier": {"value": "3000", "min": "2700", "max": "3300", "unit": ""}, "property_type": "property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": "0.4", "min": None, "max": None, "unit": "mm"}, "property_type": "ul_property"}, {"property_name": "flame rating", "modifier": {"value": "v0", "min": None, "max": None, "unit": ""}, "property_type": "ul_sub_property"}], "FILLER": [{"filler_name": ["glass fiber", "mineral"]}, {"total_load": {"value": "30", "min": "25", "max": "35"}}], "FEATURE": ["heat resistant", "impact modified"], "PROCESSING": ["coating", "blow molding"], "DELIVERY_FORM": ["granules"], "COMPETITOR_GRADE": ["novaduran 5810gn6-30"], "AUTO_CERT": [{"oem": "ford", "certs": ["wss-m4d1038", "wss-m9p14-a1"]}, {"oem": "bmw", "certs": ["mad p1 0367"]}], "RAILWAY_CERT": [{"standard": "en 45545-2", "hazard_level": ["hl2"], "req_set": ["r22"]}], "WATER_CERT": [{"standard": "acs", "temp": ["60"]}, {"standard": "nsf 61", "temp": ["100"]}], "NSF_CERT": ["nsf 61", "nsf 372"]}
# '''
# """

def create_dataset(search_query, ner_output):
    return {
        "messages": [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user", "content": search_query},
            {"role": "assistant", "content": ner_output},
        ]
    }

In [7]:
with open("data/Training_Data_27_01_25/Training_Data_27_01_25.jsonl", "w") as f:
    for key, row in td_27_01_25.iterrows():
        search_query = row['Query']
        ner_output = str(row["Output"])
        example_str = json.dumps(create_dataset(search_query, ner_output))
        f.write(example_str + "\n")

In [8]:
# Load dataset
with open("data/Training_Data_27_01_25/Training_Data_27_01_25.jsonl") as f:
    dataset = [json.loads(line) for line in f]

In [9]:
len(dataset)

61532

### Checking Data Format & Erros

In [10]:
# Format error checks
format_errors = defaultdict(int)

for idx, ex in enumerate(dataset):
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue

    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue

    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1

        if any(k not in ("role", "content", "name") for k in message):
            format_errors["message_unrecognized_key"] += 1

        if message.get("role", None) not in ("system", "user", "assistant"):
            format_errors["unrecognized_role"] += 1

        content = message.get("content", None)
        if not content or not isinstance(content, str):
            format_errors["missing_content"] += 1
            print(idx)

    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found


In [11]:
# Token counting functions
encoding = tiktoken.get_encoding("cl100k_base")

# not exact!
# simplified from https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

In [12]:
# Warnings and tokens counts
n_missing_system = 0
n_missing_user = 0
n_messages = []
convo_lens = []
assistant_message_lens = []

for ex in dataset:
    messages = ex["messages"]
    if not any(message["role"] == "system" for message in messages):
        n_missing_system += 1
    if not any(message["role"] == "user" for message in messages):
        n_missing_user += 1
    n_messages.append(len(messages))
    convo_lens.append(num_tokens_from_messages(messages))
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))

print("Num examples missing system message:", n_missing_system)
print("Num examples missing user message:", n_missing_user)
print_distribution(n_messages, "num_messages_per_example")
print_distribution(convo_lens, "num_total_tokens_per_example")
print_distribution(assistant_message_lens, "num_assistant_tokens_per_example")
n_too_long = sum(l > 4096 for l in convo_lens)
print(f"\n{n_too_long} examples may be over the 4096 token limit, they will be truncated during fine-tuning")

Num examples missing system message: 0
Num examples missing user message: 0

#### Distribution of num_messages_per_example:
min / max: 3, 3
mean / median: 3.0, 3.0
p5 / p95: 3.0, 3.0

#### Distribution of num_total_tokens_per_example:
min / max: 155, 506
mean / median: 188.43362803094325, 175.0
p5 / p95: 165.0, 230.0

#### Distribution of num_assistant_tokens_per_example:
min / max: 88, 369
mean / median: 112.15226223753494, 99.0
p5 / p95: 94.0, 149.0

0 examples may be over the 4096 token limit, they will be truncated during fine-tuning


In [13]:
# Pricing and default n_epochs estimate
MAX_TOKENS_PER_EXAMPLE = 4096

TARGET_EPOCHS = 3
MIN_TARGET_EXAMPLES = 100
MAX_TARGET_EXAMPLES = 25000
MIN_DEFAULT_EPOCHS = 1
MAX_DEFAULT_EPOCHS = 25

n_epochs = TARGET_EPOCHS
n_train_examples = len(dataset)
if n_train_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
    n_epochs = min(MAX_DEFAULT_EPOCHS, MIN_TARGET_EXAMPLES // n_train_examples)
elif n_train_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
    n_epochs = max(MIN_DEFAULT_EPOCHS, MAX_TARGET_EXAMPLES // n_train_examples)

n_billing_tokens_in_dataset = sum(min(MAX_TOKENS_PER_EXAMPLE, length) for length in convo_lens)
print(f"Dataset has ~{n_billing_tokens_in_dataset} tokens that will be charged for during training")
print(f"By default, you'll train for {n_epochs} epochs on this dataset")
print(f"By default, you'll be charged for ~{n_epochs * n_billing_tokens_in_dataset} tokens")
print("See pricing page to estimate total costs")

Dataset has ~11594698 tokens that will be charged for during training
By default, you'll train for 1 epochs on this dataset
By default, you'll be charged for ~11594698 tokens
See pricing page to estimate total costs


In [14]:
11590496*0.008/1000 # old pricing. Need to check latest pricing

92.72396800000001

In [15]:
import random
random.shuffle(dataset)
train, val = train_test_split(dataset, test_size=0.20, random_state=42)

In [16]:
len(train), len(val)

(49225, 12307)

In [17]:
def save_to_jsonl(conversations, file_path):
    with open(file_path, 'w') as file:
        for conversation in conversations:
            json_line = json.dumps(conversation)
            file.write(json_line + '\n')

In [18]:
#tain dataset
save_to_jsonl(train, 'data/Training_Data_27_01_25/train_27_01_25.jsonl')

# validate dataset
save_to_jsonl(val, 'data/Training_Data_27_01_25/validation_27_01_25.jsonl')

In [19]:
# with open(training_file_name, "r") as f:
#     ttt=f.read()

### Upload the file

In [20]:
training_file_path =  'data/Training_Data_27_01_25/train_27_01_25.jsonl'
validation_file_path = 'data/Training_Data_27_01_25/validation_27_01_25.jsonl'
training_file_name = 'train_27_01_25.jsonl'
validation_file_name = 'validation_27_01_25.jsonl'

In [21]:
# with open(training_file_name) as f:
#     training_file = [json.loads(line) for line in f]
    
# with open(validation_file_name) as f:
#     validation_file = f.read()

In [22]:
training_response = openai.File.create(
    file=open(training_file_path, "r"), purpose="fine-tune", user_provided_filename=training_file_name
)
training_file_id = training_response["id"]

print("Training file id:", training_file_id)



validation_response = openai.File.create(
    file=open(validation_file_path, "r"), purpose="fine-tune", user_provided_filename=validation_file_name
)
validation_file_id = validation_response["id"]

print("Validation file id:", validation_file_id)

Training file id: file-a3b0d8eea34c469391fe9a6d3bc3c84a
Validation file id: file-4866d639a96d492ab99e606dcb80c101
